# Quantitative analysis: XGBoost and neural network

Run `.venv/Scripts/python run_quantitative_analysis.py` from `XAI_01` to refresh the outputs. Both models use the same untouched, stratified 80/20 holdout from the balanced 2015 BRFSS survey. The target is **reported prediabetes or diabetes**, not future onset. ROC AUC, calibration, and subgroup error rates describe prediction quality; the Quantus tests examine explanations of the positive-class probability. All figures also have editable SVG versions in `analysis_outputs`.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

OUT = Path('analysis_outputs')
results = json.loads((OUT / 'results.json').read_text(encoding='utf-8'))
pred = pd.read_csv(OUT / 'holdout_predictions.csv')
slices = pd.read_csv(OUT / 'error_slices.csv')
quantus_scores = pd.read_csv(OUT / 'quantus_scores.csv')
radar_scores = pd.read_csv(OUT / 'evaluation_radar.csv')
assert len(radar_scores) == 20
edge_cases = pd.read_csv(OUT / 'edge_cases.csv')
attrs = np.load(OUT / 'attributions.npz')
features = list(results['case']['features'])
assert len(pred) == results['test_rows']
assert len(quantus_scores) == 640 and quantus_scores.value.notna().all()
display(pd.DataFrame([{'records': results['data_rows'], 'train': results['train_rows'],
                       'holdout': results['test_rows'], 'features': len(features),
                       'explained records': len(attrs['indices'])}]))

## Model validation

In [ ]:
rows = []
for model, m in results['model_metrics'].items():
    rows.append({'model': model, 'ROC AUC': m['auc'], 'accuracy': m['accuracy'],
                 'positive recall': m['sensitivity'], 'negative recall': m['specificity'],
                 'missed positives': m['confusion_matrix'][1][0], 'Brier': m['brier']})
display(pd.DataFrame(rows).round(3))
auc = results['auc_comparison']
print(f"AUC gap (XGBoost − NN): {auc['difference']:+.4f}; 95% paired bootstrap interval {auc['ci_95'][0]:+.4f} to {auc['ci_95'][1]:+.4f}")

![ROC, balanced-sample calibration, and confusion counts](analysis_outputs/01_model_validation.png)

## Error slices and edge cases

In [ ]:
view = slices.copy()
view['positive_cases'] = (view.n * view.observed_positive_rate).round().astype(int)
display(view[['group', 'model', 'n', 'positive_cases', 'false_negative_rate']]
        .sort_values(['group', 'model']).round(3))
display(edge_cases[['model', 'case_type', 'holdout_index', 'label', 'score',
                    'Age', 'BMI', 'HighBP', 'GenHlth']].head(12).round(3))

![Missed positive cases by group](analysis_outputs/02_error_slices.png)

## Global feature patterns

Mean absolute attributions on 40 holdout records. SHAP explanations for both models target **positive-class probability** (Gradient SHAP for the NN is approximate); LIME coefficients are weights from a local surrogate. Compare ranks within each panel, since SHAP and LIME have different meanings. These 40 records provide an exploratory summary, not a population estimate.


In [ ]:
global_rows = []
for model in ('XGBoost', 'NN'):
    for method in ('SHAP', 'LIME'):
        mean_abs = np.abs(attrs[f'{model}_{method}']).mean(axis=0)
        for rank, j in enumerate(np.argsort(mean_abs)[::-1][:5], 1):
            global_rows.append({'model': model, 'method': method, 'rank': rank,
                                'feature': features[j], 'mean_abs': mean_abs[j]})
display(pd.DataFrame(global_rows).round(3))

![Global SHAP and LIME attributions](analysis_outputs/03_global_attributions.png)

## Quantus explanation checks

Faithfulness correlation measures whether replacing a credited feature subset changes the positive-class probability in the predicted direction. Replacement values come from observed training records; two donor seeds expose baseline sensitivity. Sparseness and complexity describe concentration, not correctness. Keep their raw distributions separate from the five-spoke profile below.


In [ ]:
summary = (quantus_scores.groupby(['model', 'method', 'metric', 'replicate'])
           .value.median().unstack(['metric', 'replicate']))
display(summary.round(3))
paired = []
for model in ('XGBoost', 'NN'):
    pair = (quantus_scores[(quantus_scores.model == model) &
                           (quantus_scores.metric == 'faithfulness_correlation')]
            .groupby(['holdout_index', 'method']).value.mean().unstack())
    delta = (pair.SHAP - pair.LIME).dropna().to_numpy()
    rng = np.random.default_rng(99)
    boots = [rng.choice(delta, len(delta), replace=True).mean() for _ in range(2000)]
    low, high = np.quantile(boots, [.025, .975])
    paired.append({'model': model, 'mean SHAP − LIME': delta.mean(),
                   '95% CI low': low, '95% CI high': high})
display(pd.DataFrame(paired).round(3))

![Quantus faithfulness, sparseness, and complexity](analysis_outputs/04_quantus_scores.png)

## Feature flipping

Quantus selects the highest positive contributions first and assigns values from observed donor records. A selected feature can already equal its donor value. A random feature order is the control; faster score decline supports the explanation on this test.


In [ ]:
flipping = pd.read_csv(OUT / 'feature_flipping.csv')
curve = (flipping.groupby(['model', 'method', 'features_replaced'])
         .positive_score.mean().reset_index())
flip_rows = []
for (model, method), group in curve.groupby(['model', 'method']):
    group = group.sort_values('features_replaced')
    flip_rows.append({'model': model, 'method': method,
                      'score after 5 replacements': group.loc[group.features_replaced == 5,
                                                               'positive_score'].iloc[0],
                      'area under score curve': np.trapezoid(group.positive_score,
                                                             group.features_replaced) / 21})
flip_summary = pd.DataFrame(flip_rows)
display(flip_summary.round(3))


![Quantus feature flipping with a random-order control](analysis_outputs/07_feature_flipping.png)


## Explanation stability

Quantus MaxSensitivity checks the largest explanation change after one valid unit edit to BMI or poor-health days. Lower scores are steadier.


In [ ]:
stability = pd.read_csv(OUT / 'explanation_stability.csv')
display(stability.groupby(['model', 'method']).max_sensitivity
        .agg(cases_scored='count', median='median').round(3))


![Quantus explanation stability](analysis_outputs/08_explanation_stability.png)


## Explanation agreement

Compare the five strongest features for each person across models and methods. Agreement is descriptive, not proof that either explanation is correct.


In [ ]:
agreement = pd.read_csv(OUT / 'explanation_agreement.csv')
mean_overlap = (agreement.groupby(['first', 'second']).top_five_overlap.mean()
                .unstack().round(2))
display(mean_overlap)


![Top-five feature agreement across models and methods](analysis_outputs/09_explanation_agreement.png)


## Five-spoke evaluation profile

This is the **radar chart** suggested by the Quantus figure in Lecture 6. Each spoke is derived from the tests above and below; it is a compact reading aid, **not a composite quality score**. Faithfulness correlation is mapped from −1…1 to 0…1. Flipping win rate is the fraction of paired case/donor runs where SHAP or LIME lowers the positive-class score faster than random order over the first five replacements. Stability is `1 / (1 + median MaxSensitivity)`. Sparseness and cross-model top-five overlap are descriptive; high values do not prove fidelity. The exact values, transformations, definitions, and sample sizes are in `evaluation_radar.csv`.


In [ ]:
radar = pd.read_csv(OUT / 'evaluation_radar.csv')
display(radar[['model', 'method', 'dimension', 'raw_value', 'plotted_value', 'n']].round(3))
assert len(radar) == 20 and radar.plotted_value.between(0, 1).all()


![Five-spoke evaluation profile](analysis_outputs/10_evaluation_radar.png)


## One de-identified record

Selected before reviewing explanations: a positive record with XGBoost score nearest 0.75.

In [ ]:
case = results['case']
display(pd.DataFrame([{'holdout index': case['holdout_index'], 'label': case['label'],
                       'XGBoost score': case['scores']['XGBoost'],
                       'NN score': case['scores']['NN']}]).round(3))
display(pd.DataFrame([{'feature': k, 'recorded value': case['features'][k]}
                      for k in ('HighBP', 'HighChol', 'BMI', 'PhysActivity', 'GenHlth', 'Age')]))

![Local model score contributions](analysis_outputs/06_patient_explanation.png)

![Technical SHAP and LIME view](analysis_outputs/05_local_case.png)

**Limits:** This is a balanced, cross-sectional survey, so scores are not personal future-risk estimates. Donor replacement can create unusual feature combinations. The explanation subset and stability subset are small; treat method differences as exploratory. Attribution agreement, sparseness, and a large radar area do not establish truth, causality, or clinical safety.
